# PHYTORA · Pipeline B, part 2: Grad-CAM and ONNX export
Continues the training notebook (MobileNetV3-Small, 38 PlantVillage classes).

**What this notebook produces**
1. **Grad-CAM** on 10 test images (Corn Northern Leaf Blight, Potato healthy, random ones) + a quantitative "does the model look at the leaf?" check.
2. **ONNX export** with a parity check (PyTorch vs ONNX Runtime), CPU latency, and a full test-set comparison.
3. A ready-to-hand-over folder for the Full-Stack lead: `model.onnx`, `classes.json`, `preprocess.json`, `reference_inference.py`, `export_report.json`.

**How to run it**
- *Same session as the training notebook:* append these cells at the end and run them.
- *New notebook:* Settings → **GPU on**, **Internet on**; *Add Input* → the PlantVillage dataset **and** the output of your training notebook (or the zip uploaded as a dataset). The checkpoint and CSVs are found automatically.

## 0. Setup

In [ ]:
import os, json, time, copy, glob, shutil, subprocess, sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from PIL import Image
from tqdm.auto import tqdm

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from sklearn.metrics import accuracy_score, f1_score

SEED = 42
NUM_WORKERS = 2
N_EVAL = 200            # images used for the quantitative Grad-CAM check
FULL_TEST = True        # compare PyTorch vs ONNX on the whole test set (a few minutes)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)

rng = np.random.RandomState(SEED)
torch.manual_seed(SEED)


def find_file(name):
    """Look for a file in /kaggle/working then /kaggle/input (skipping the big image folders)."""
    for root in ("/kaggle/working", "/kaggle/input"):
        for dirpath, dirnames, filenames in os.walk(root):
            dirnames[:] = [d for d in dirnames if d not in ("color", "grayscale", "segmented")]
            if name in filenames:
                return Path(dirpath) / name
    raise FileNotFoundError(f"{name} not found: add your training notebook output as an input.")


CKPT = find_file("mobilenet_v3_small_best.pt")
TEST_CSV = find_file("test.csv")
print("Checkpoint :", CKPT)
print("Test CSV   :", TEST_CSV)

DATA_ROOT = Path("/kaggle/input/datasets/abdallahalidev/plantvillage-dataset/color")
if not DATA_ROOT.exists():
    candidates = [p for p in Path("/kaggle/input").rglob("color") if p.is_dir()]
    if not candidates:
        raise FileNotFoundError("Folder 'color' not found: add the PlantVillage dataset as an input.")
    DATA_ROOT = candidates[0]
print("Images     :", DATA_ROOT)

WORK = Path("/kaggle/working")
EXPORT_DIR = WORK / "export"
FIG_DIR = WORK / "figures"
for d in (EXPORT_DIR, FIG_DIR):
    d.mkdir(parents=True, exist_ok=True)

In [ ]:
import os, json, time, copy, glob, shutil, subprocess, sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from PIL import Image
from tqdm.auto import tqdm

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from sklearn.metrics import accuracy_score, f1_score

SEED = 42
NUM_WORKERS = 2
N_EVAL = 200            # images used for the quantitative Grad-CAM check
FULL_TEST = True        # compare PyTorch vs ONNX on the whole test set (a few minutes)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)

rng = np.random.RandomState(SEED)
torch.manual_seed(SEED)


def find_file(name):
    """Look for a file in /kaggle/working then /kaggle/input (skipping the big image folders)."""
    for root in ("/kaggle/working", "/kaggle/input"):
        for dirpath, dirnames, filenames in os.walk(root):
            dirnames[:] = [d for d in dirnames if d not in ("color", "grayscale", "segmented")]
            if name in filenames:
                return Path(dirpath) / name
    raise FileNotFoundError(f"{name} not found: add your training notebook output as an input.")


CKPT = find_file("mobilenet_v3_small_best.pt")
TEST_CSV = find_file("test.csv")
print("Checkpoint :", CKPT)
print("Test CSV   :", TEST_CSV)

DATA_ROOT = Path("/kaggle/input/datasets/abdallahalidev/plantvillage-dataset/color")
if not DATA_ROOT.exists():
    candidates = [p for p in Path("/kaggle/input").rglob("color") if p.is_dir()]
    if not candidates:
        raise FileNotFoundError("Folder 'color' not found: add the PlantVillage dataset as an input.")
    DATA_ROOT = candidates[0]
print("Images     :", DATA_ROOT)

WORK = Path("/kaggle/working")
EXPORT_DIR = WORK / "export"
FIG_DIR = WORK / "figures"
for d in (EXPORT_DIR, FIG_DIR):
    d.mkdir(parents=True, exist_ok=True)

## 1. Grad-CAM: is the model looking at the leaf?
Grad-CAM highlights the image regions that pushed the model toward its predicted class. The EDA showed that background and lighting already carry class information, so we want evidence that the model focuses on the **leaf**, not the backdrop.

In [ ]:
class GradCAM:
    """Grad-CAM on the last convolutional block of MobileNetV3."""

    def __init__(self, net, target_layer):
        self.net = net.eval()
        self.acts, self.grads = None, None
        target_layer.register_forward_hook(self._on_forward)

    def _on_forward(self, module, inputs, output):
        self.acts = output
        output.register_hook(self._on_grad)   # tensor hook: safe with in-place activations

    def _on_grad(self, grad):
        self.grads = grad

    def __call__(self, x, class_idx=None):
        self.net.zero_grad()
        logits = self.net(x)
        if class_idx is None:
            class_idx = int(logits.argmax(1).item())
        logits[0, class_idx].backward()
        w = self.grads.mean(dim=(2, 3), keepdim=True)
        cam = torch.relu((w * self.acts).sum(dim=1, keepdim=True))
        cam = torch.nn.functional.interpolate(cam, size=x.shape[-2:], mode="bilinear", align_corners=False)[0, 0]
        cam = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)
        prob = float(torch.softmax(logits, 1)[0, class_idx])
        return cam.detach().cpu().numpy(), class_idx, prob


cam_engine = GradCAM(cam_model, cam_model.features[-1])


def explain(img):
    x = eval_transform(img.convert("RGB")).unsqueeze(0).to(DEVICE)
    return cam_engine(x)


def overlay(img, cam, alpha=0.45):
    base = np.asarray(img.convert("RGB").resize((IMG_SIZE, IMG_SIZE)), dtype=np.float32) / 255.0
    heat = matplotlib.colormaps["jet"](cam)[..., :3]
    return np.clip((1 - alpha) * base + alpha * heat, 0, 1)

In [ ]:
# ---- choose ~10 test images: Corn NLB (errors first), Potato healthy, and random ones
def pick(class_name, n_wrong, n_right):
    sub = test_df[test_df["label"] == class_name].reset_index(drop=True)
    if sub.empty:
        print("Class not found:", class_name)
        return sub
    sub["pred"], _ = predict_batch(sub["path"].tolist())
    wrong, right = sub[sub["pred"] != sub["class_idx"]], sub[sub["pred"] == sub["class_idx"]]
    print(f"{class_name}: {len(wrong)} errors / {len(sub)} test images")
    parts = [wrong.sample(min(n_wrong, len(wrong)), random_state=SEED),
             right.sample(min(n_right, len(right)), random_state=SEED)]
    return pd.concat(parts)


corn = pick("Corn_(maize)___Northern_Leaf_Blight", 2, 1).assign(note="Corn NLB")
potato = pick("Potato___healthy", 1, 2).assign(note="Potato healthy")
chosen = pd.concat([corn, potato])
others = test_df[~test_df["path"].isin(chosen["path"])].sample(10 - len(chosen), random_state=SEED).assign(note="random")
sel = pd.concat([chosen, others]).reset_index(drop=True)
print("Selected images:", len(sel))

In [ ]:
n = len(sel)
rows = (n + 1) // 2
fig, axes = plt.subplots(rows, 4, figsize=(15, 3.8 * rows))
axes = np.atleast_2d(axes)
for ax in axes.ravel():
    ax.axis("off")

for i, r in sel.iterrows():
    img = Image.open(DATA_ROOT / r["path"]).convert("RGB")
    cam, pred, prob = explain(img)
    ok = "OK" if pred == r["class_idx"] else "WRONG"
    ax_img, ax_cam = axes[i // 2, (i % 2) * 2], axes[i // 2, (i % 2) * 2 + 1]
    ax_img.imshow(img)
    ax_img.set_title(f"[{r['note']}] {ok}\ntrue: {short(r['label'])}", fontsize=8)
    ax_cam.imshow(overlay(img, cam))
    ax_cam.set_title(f"pred: {short(CLASSES[pred])}\np = {prob:.2f}", fontsize=8)
plt.tight_layout()
plt.savefig(FIG_DIR / "gradcam_examples.png", dpi=130)
plt.show()

**How to read it:** red/yellow = regions that drive the decision. A good model concentrates on the leaf surface and lesions. If the heat sits on the corners or the backdrop, the model is using a shortcut, and that must be written up as a limitation.

In [ ]:
# ---- Quantitative check (needs the "segmented" folder of the Kaggle dataset, otherwise skipped)
SEG_ROOT = DATA_ROOT.parent / "segmented"


def find_mask(rel_path):
    p = Path(rel_path)
    cls_dir = SEG_ROOT / p.parent
    if not cls_dir.exists():
        return None
    for suffix in ("_final_masked.jpg", "_final_masked.JPG", ".JPG", ".jpg"):
        cand = cls_dir / f"{p.stem}{suffix}"
        if cand.exists():
            return cand
    hits = glob.glob(glob.escape(str(cls_dir / p.stem)) + "*")
    return Path(hits[0]) if hits else None


records = []
for _, r in tqdm(test_df.sample(min(N_EVAL, len(test_df)), random_state=SEED).iterrows(), total=min(N_EVAL, len(test_df))):
    mp = find_mask(r["path"])
    if mp is None:
        continue
    img = Image.open(DATA_ROOT / r["path"]).convert("RGB")
    mask = np.asarray(Image.open(mp).convert("L").resize((IMG_SIZE, IMG_SIZE), Image.NEAREST)) > 10
    area = float(mask.mean())
    if area < 0.02:
        continue
    cam, pred, prob = explain(img)
    records.append({"correct": pred == r["class_idx"], "leaf_area_share": area,
                    "cam_share_on_leaf": float(cam[mask].sum() / (cam.sum() + 1e-8))})

if not records:
    print("No segmented masks found: quantitative check skipped (visual check above still applies).")
else:
    q = pd.DataFrame(records)
    q["focus_ratio"] = q["cam_share_on_leaf"] / q["leaf_area_share"]
    print(f"Images analysed: {len(q)}")
    summary = pd.DataFrame({
        "leaf area share (chance level)": [q["leaf_area_share"].mean()],
        "Grad-CAM share on leaf": [q["cam_share_on_leaf"].mean()],
        "focus ratio (>1 = focuses on leaf)": [q["focus_ratio"].mean()],
        "% images with ratio > 1": [100 * (q["focus_ratio"] > 1).mean()],
    }).round(3)
    display(summary)
    display(q.groupby("correct")[["cam_share_on_leaf", "focus_ratio"]].mean().round(3))
    q.to_csv(FIG_DIR / "gradcam_leaf_focus.csv", index=False)

## 2. ONNX export
- Export with a **dynamic batch axis**; the output is **logits** (the API applies the softmax).
- **Parity check:** PyTorch vs ONNX Runtime on real test images.
- **CPU latency** for one image (what the API will do).
- Optional **full test-set** comparison to prove the exported model scores the same.

In [ ]:
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "onnx", "onnxruntime", "onnxscript"], check=False)
import onnx
import onnxruntime as ort
print("torch", torch.__version__, "| onnx", onnx.__version__, "| onnxruntime", ort.__version__)

In [ ]:
export_model = load_model().cpu()          # fresh, clean copy (no Grad-CAM hooks)
dummy = torch.randn(1, 3, IMG_SIZE, IMG_SIZE)
ONNX_PATH = EXPORT_DIR / "model.onnx"
OPSET = 17


def export_to_onnx():
    kwargs = dict(input_names=["input"], output_names=["logits"],
                  dynamic_axes={"input": {0: "batch"}, "logits": {0: "batch"}}, opset_version=OPSET)
    for extra in ({"dynamo": False}, {}):   # classic exporter first, then the default one
        try:
            torch.onnx.export(export_model, dummy, str(ONNX_PATH), **kwargs, **extra)
            return extra
        except Exception as e:
            print(f"Export attempt {extra} failed: {type(e).__name__}: {str(e)[:200]}")
    raise RuntimeError("ONNX export failed with both exporters")


used = export_to_onnx()
onnx.checker.check_model(onnx.load(str(ONNX_PATH)))
print(f"Exported {ONNX_PATH.name} ({ONNX_PATH.stat().st_size / 1e6:.1f} MB), exporter options: {used or 'default'}; checker OK")

In [ ]:
# ---- Parity check on 32 real test images
sess = ort.InferenceSession(str(ONNX_PATH), providers=["CPUExecutionProvider"])
batch_paths = test_df.sample(32, random_state=SEED)["path"].tolist()
x = torch.stack([eval_transform(Image.open(DATA_ROOT / p).convert("RGB")) for p in batch_paths])
with torch.no_grad():
    ref = export_model(x).numpy()
got = sess.run(None, {"input": x.numpy()})[0]

max_diff = float(np.abs(ref - got).max())
same_top1 = bool((ref.argmax(1) == got.argmax(1)).all())
print(f"Max |logit difference| PyTorch vs ONNX : {max_diff:.2e}")
print(f"Same predicted class on all 32 images  : {same_top1}")
assert max_diff < 1e-3 and same_top1, "ONNX export is not faithful to the PyTorch model"

# batch axis really dynamic?
print("Batch of 1 ->", sess.run(None, {"input": x[:1].numpy()})[0].shape, "| batch of 8 ->", sess.run(None, {"input": x[:8].numpy()})[0].shape)

In [ ]:
# ---- CPU latency, single image
def bench(fn, n=100, warmup=10):
    for _ in range(warmup):
        fn()
    times = []
    for _ in range(n):
        t0 = time.perf_counter()
        fn()
        times.append((time.perf_counter() - t0) * 1000)
    return {"mean_ms": float(np.mean(times)), "p95_ms": float(np.percentile(times, 95))}


x1 = x[:1]
with torch.no_grad():
    lat_torch = bench(lambda: export_model(x1))
lat_onnx = bench(lambda: sess.run(None, {"input": x1.numpy()}))
print(f"CPU ({os.cpu_count()} cores) | PyTorch: {lat_torch['mean_ms']:.1f} ms (p95 {lat_torch['p95_ms']:.1f}) | "
      f"ONNX Runtime: {lat_onnx['mean_ms']:.1f} ms (p95 {lat_onnx['p95_ms']:.1f})")
print("Model inference only: add preprocessing (decode + resize) for the end-to-end API latency.")

In [ ]:
# ---- Whole test set: PyTorch (GPU/CPU) vs ONNX Runtime
class TestDataset(Dataset):
    def __init__(self, df):
        self.paths, self.y = df["path"].tolist(), df["class_idx"].astype(int).tolist()

    def __len__(self):
        return len(self.y)

    def __getitem__(self, i):
        return eval_transform(Image.open(DATA_ROOT / self.paths[i]).convert("RGB")), self.y[i]


full = {}
if FULL_TEST:
    loader = DataLoader(TestDataset(test_df), batch_size=64, shuffle=False, num_workers=NUM_WORKERS)
    y_true, p_torch, p_onnx = [], [], []
    with torch.no_grad():
        for xb, yb in tqdm(loader, desc="test set"):
            p_torch.append(model(xb.to(DEVICE)).argmax(1).cpu().numpy())
            p_onnx.append(sess.run(None, {"input": xb.numpy()})[0].argmax(1))
            y_true.append(yb.numpy())
    y_true, p_torch, p_onnx = map(np.concatenate, (y_true, p_torch, p_onnx))
    full = {
        "images": int(len(y_true)),
        "pytorch_acc": float(accuracy_score(y_true, p_torch)),
        "pytorch_macro_f1": float(f1_score(y_true, p_torch, average="macro", zero_division=0)),
        "onnx_acc": float(accuracy_score(y_true, p_onnx)),
        "onnx_macro_f1": float(f1_score(y_true, p_onnx, average="macro", zero_division=0)),
        "prediction_agreement": float((p_torch == p_onnx).mean()),
    }
    display(pd.Series(full).round(5).to_frame("value"))

## 3. Hand-over package for the Full-Stack lead
`reference_inference.py` contains the **exact preprocessing** to reuse in `backend/app/inference.py`. A mismatch in preprocessing between training and API is the most common cause of "the model works in the notebook but not in the app", so we verify this file against PyTorch below.

In [ ]:
REFERENCE_PY = '"""\nPHYTORA - Pipeline B : reference inference with ONNX Runtime.\n\nCopy this logic into backend/app/inference.py. Preprocessing is IDENTICAL to training:\nRGB -> PIL bilinear resize -> /255 -> (x - mean) / std -> NCHW float32.\nThe model outputs logits: softmax is applied here.\n"""\nimport json\nfrom pathlib import Path\n\nimport numpy as np\nimport onnxruntime as ort\nfrom PIL import Image\n\n\nclass DiseaseClassifier:\n    def __init__(self, model_dir):\n        d = Path(model_dir)\n        self.session = ort.InferenceSession(str(d / "model.onnx"), providers=["CPUExecutionProvider"])\n        cfg = json.load(open(d / "preprocess.json", encoding="utf-8"))\n        class_to_idx = json.load(open(d / "classes.json", encoding="utf-8"))\n        self.classes = [c for c, _ in sorted(class_to_idx.items(), key=lambda kv: kv[1])]\n        self.size = tuple(cfg["resize"])\n        self.mean = np.array(cfg["mean"], dtype=np.float32).reshape(1, 1, 3)\n        self.std = np.array(cfg["std"], dtype=np.float32).reshape(1, 1, 3)\n        self.input_name = cfg["input_name"]\n\n    def preprocess(self, image):\n        img = image.convert("RGB").resize(self.size, Image.BILINEAR)\n        arr = (np.asarray(img, dtype=np.float32) / 255.0 - self.mean) / self.std\n        return arr.transpose(2, 0, 1)[None].astype(np.float32)\n\n    def predict(self, image, top_k=5):\n        logits = self.session.run(None, {self.input_name: self.preprocess(image)})[0][0]\n        e = np.exp(logits - logits.max())\n        probs = e / e.sum()\n        order = probs.argsort()[::-1][:top_k]\n        return [{"label": self.classes[i], "score": float(probs[i])} for i in order]\n'

json.dump({c: i for i, c in enumerate(CLASSES)}, open(EXPORT_DIR / "classes.json", "w", encoding="utf-8"),
          indent=2, ensure_ascii=False)
json.dump({
    "model": ckpt["model"], "onnx_opset": OPSET,
    "input_name": "input", "output_name": "logits",
    "input_shape": ["batch", 3, IMG_SIZE, IMG_SIZE], "dtype": "float32", "layout": "NCHW",
    "channel_order": "RGB", "resize": [IMG_SIZE, IMG_SIZE], "resize_method": "PIL bilinear",
    "scale": "pixels / 255.0", "mean": MEAN, "std": STD,
    "output": "logits (one per class, order = classes.json); apply softmax in the API",
    "num_classes": NUM_CLASSES,
}, open(EXPORT_DIR / "preprocess.json", "w", encoding="utf-8"), indent=2, ensure_ascii=False)
open(EXPORT_DIR / "reference_inference.py", "w", encoding="utf-8").write(REFERENCE_PY)
print("Written:", sorted(p.name for p in EXPORT_DIR.iterdir()))

In [ ]:
# ---- Verify the reference inference code against PyTorch on 20 test images
import importlib.util
spec = importlib.util.spec_from_file_location("reference_inference", EXPORT_DIR / "reference_inference.py")
ref_mod = importlib.util.module_from_spec(spec)
spec.loader.exec_module(ref_mod)
clf = ref_mod.DiseaseClassifier(EXPORT_DIR)

paths = test_df.sample(20, random_state=1)["path"].tolist()
max_prob_diff, agree = 0.0, 0
for p in paths:
    img = Image.open(DATA_ROOT / p)
    top = clf.predict(img, top_k=1)[0]
    with torch.no_grad():
        probs = torch.softmax(export_model(eval_transform(img.convert("RGB")).unsqueeze(0)), 1)[0]
    max_prob_diff = max(max_prob_diff, abs(top["score"] - float(probs.max())))
    agree += int(top["label"] == CLASSES[int(probs.argmax())])
print(f"reference_inference.py vs PyTorch: same class on {agree}/20 images | max |prob difference| = {max_prob_diff:.2e}")
assert agree == 20 and max_prob_diff < 1e-3

print("\nExample output for the API:")
print(json.dumps(clf.predict(Image.open(DATA_ROOT / paths[0]), top_k=3), indent=2))

In [ ]:
report = {
    "model": ckpt["model"], "classes": NUM_CLASSES, "onnx_opset": OPSET,
    "onnx_size_mb": round(ONNX_PATH.stat().st_size / 1e6, 2),
    "parity": {"max_abs_logit_diff": max_diff, "same_top1_on_32_images": same_top1},
    "cpu_latency_single_image_ms": {"pytorch": lat_torch, "onnxruntime": lat_onnx, "cpu_cores": os.cpu_count()},
    "full_test_set": full,
    "versions": {"torch": torch.__version__, "onnx": onnx.__version__, "onnxruntime": ort.__version__},
}
json.dump(report, open(EXPORT_DIR / "export_report.json", "w"), indent=2)

bundle = WORK / "bundle"
bundle.mkdir(exist_ok=True)
shutil.copytree(EXPORT_DIR, bundle / "export", dirs_exist_ok=True)
shutil.copytree(FIG_DIR, bundle / "figures", dirs_exist_ok=True)
archive = shutil.make_archive(str(WORK / "phytora_pipeline_b_export"), "zip", bundle)
shutil.rmtree(bundle)
print("Archive:", archive, f"({Path(archive).stat().st_size / 1e6:.1f} MB)")
print(json.dumps(report, indent=2))

### Next
1. Send `phytora_pipeline_b_export.zip` (`export/` folder) to the Full-Stack lead: replace the mock model with `model.onnx` and reuse `reference_inference.py`.
2. Put `model.onnx` under DVC and register it in MLflow (MLOps lead).
3. Write the Grad-CAM findings in the report (what the model looks at, which classes are confused, limits).
4. Evaluate on photos taken **outside** PlantVillage to measure the lab-to-field gap.